# Sales Data Analysis Dashboard
Pandas + NumPy + Matplotlib. Run every cell from top to bottom with **Shift + Enter** (or Kernel -> Restart & Run All).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Dataset
This cell creates `sales_data.csv` (sample data) if it does not exist. To use your own dataset, put your file in the same folder with the same name and column names.

In [ ]:
if not os.path.exists('sales_data.csv'):
    rng = np.random.default_rng(42)
    N = 5000

    catalog = {
        "Electronics": {"Laptop": 55000, "Smartphone": 25000, "Headphones": 3000, "Smartwatch": 8000},
        "Clothing": {"T-Shirt": 600, "Jeans": 1800, "Jacket": 3500, "Sneakers": 2800},
        "Home": {"Mixer Grinder": 3200, "Bedsheet": 1200, "Lamp": 900, "Air Fryer": 6500},
        "Books": {"Python Book": 700, "Novel": 350, "Textbook": 900, "Comics": 250},
    }
    rows = [(c, p, price) for c, items in catalog.items() for p, price in items.items()]
    pick = rng.integers(0, len(rows), N)

    dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 730, N), unit="D")
    month_boost = 1 + 0.4 * np.isin(dates.month, [10, 11, 12])   # festive-season bump
    quantity = np.clip(rng.poisson(2 * month_boost), 1, None)

    df = pd.DataFrame({
        "order_id": np.arange(1001, 1001 + N),
        "order_date": dates.strftime("%Y-%m-%d"),
        "region": rng.choice(["North", "South", "East", "West"], N, p=[0.25, 0.30, 0.15, 0.30]),
        "category": [rows[i][0] for i in pick],
        "product": [rows[i][1] for i in pick],
        "unit_price": [rows[i][2] for i in pick],
        "quantity": quantity,
        "discount_pct": rng.choice([0, 5, 10, 15, 20], N, p=[0.4, 0.2, 0.2, 0.1, 0.1]),
        "customer_age": rng.integers(18, 65, N).astype(float),
    })

    # Add some dirty data on purpose so the cleaning step in dashboard.py is meaningful
    df.loc[rng.choice(N, 60, replace=False), "customer_age"] = np.nan
    df.loc[rng.choice(N, 25, replace=False), "region"] = None
    df = pd.concat([df, df.sample(30, random_state=1)], ignore_index=True)   # duplicate rows

    df.to_csv("sales_data.csv", index=False)
    print(f"Created sales_data.csv with {len(df)} rows")
else:
    print('sales_data.csv already exists, using it.')

## 2. Load the data

In [ ]:
df = pd.read_csv('sales_data.csv')
print('Raw shape:', df.shape)
df.head()

## 3. Clean the data
Remove duplicate rows and fill missing values.

In [ ]:
print('Missing values before:\n', df.isna().sum(), '\n')
df = df.drop_duplicates()
df['region'] = df['region'].fillna('Unknown')
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].median())
df['order_date'] = pd.to_datetime(df['order_date'])
print('Clean shape:', df.shape)

## 4. Feature engineering

In [ ]:
df['revenue'] = df['unit_price'] * df['quantity'] * (1 - df['discount_pct'] / 100)
df['month'] = df['order_date'].dt.to_period('M')
df['weekday'] = df['order_date'].dt.day_name()
df['age_group'] = pd.cut(df['customer_age'], bins=[17, 25, 35, 45, 65], labels=['18-25', '26-35', '36-45', '46-65'])
df.head()

## 5. Analysis

In [ ]:
monthly = df.groupby('month')['revenue'].sum()
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
by_region = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
top_products = df.groupby('product')['revenue'].sum().nlargest(8).sort_values()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
by_weekday = df.groupby('weekday')['revenue'].sum().reindex(weekday_order)
growth = monthly.pct_change().dropna() * 100
corr = df[['unit_price', 'quantity', 'discount_pct', 'customer_age', 'revenue']].corr()

print(f"Total revenue        : Rs {df['revenue'].sum():,.0f}")
print(f"Average order value  : Rs {df['revenue'].mean():,.0f}")
print(f"Best category        : {by_category.index[0]}")
print(f"Best region          : {by_region.index[0]}")
print(f"Best month           : {monthly.idxmax()}")
print(f"Best weekday         : {by_weekday.idxmax()}")
print(f"Avg monthly growth   : {np.mean(growth):.2f}%")

## 6. Charts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly.index.astype(str), monthly.values, marker='o', color='#2a6fdb')
ax.set_title('Monthly Revenue Trend'); ax.set_ylabel('Revenue (Rs)')
ax.tick_params(axis='x', rotation=60); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(by_category.values, labels=by_category.index, autopct='%1.1f%%', startangle=90)
ax.set_title('Revenue Share by Category')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(by_region.index, by_region.values, color='#3aa17e')
ax.set_title('Revenue by Region'); ax.set_ylabel('Revenue (Rs)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top_products.index, top_products.values, color='#e07b39')
ax.set_title('Top 8 Products by Revenue')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([d[:3] for d in weekday_order], by_weekday.values, color='#8e5bd6')
ax.set_title('Revenue by Weekday')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Correlation Heatmap'); fig.colorbar(im)
plt.tight_layout(); plt.show()

## 7. Conclusions (write your own)
- Which category and region earn the most?
- Which months are strongest, and why might that be?
- What would you do with this data in a real business?